[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/solutions/Lab0_Search_Spectrum_Solutions.ipynb)


# Lab 0 SOLUTIONS: The Search Spectrum — From Exact Match to RAG
## CCI Session 6

**Duration:** 30 minutes

### Clinical Scenario
> A KHCC clinician searches through oncology patient notes to answer clinical questions.
> We progress through 7 retrieval methods — each one solves a limitation of the previous.

### Objectives
- Implement **exact**, **regex**, and **fuzzy** text search
- Build **TF-IDF** and **BM25** lexical retrieval indices
- Create a **semantic search** engine with free sentence embeddings
- Assemble an **oversimplified RAG** pipeline (retrieve → augment → generate)

### The Search Spectrum
```
Exact ──► Regex ──► Fuzzy ──► TF-IDF ──► BM25 ──► Semantic ──► RAG
◄── more literal                                    more semantic ──►
◄── faster                                               slower ──►
◄── no model needed                             model required ──►
```


In [ ]:
!pip install -q rapidfuzz rank_bm25 scikit-learn sentence-transformers openai pandas


In [ ]:
# Synthetic KHCC oncology notes (15 notes)
corpus = [
    'Patient A, 4yo male. Wilms tumor (nephroblastoma) stage II favorable histology. '
    'Vincristine 1.5 mg/m2 IV and actinomycin-D per COG AREN0532. Post-nephrectomy week 1.',
    'Patient B, 7yo female. B-cell precursor ALL. Induction: dexamethasone, vincristine, '
    'PEG-asparaginase, doxorubicin. Bone marrow 85% blasts at diagnosis.',
    'Patient C, 12yo male. Hodgkin lymphoma stage IIB. ABVD: adriamycin 25 mg/m2, '
    'bleomycin, vinblastine, dacarbazine. Biopsy confirmed Reed-Sternberg cells.',
    'Patient D, 5yo female. Medulloblastoma post-resection. Craniospinal irradiation '
    '36 Gy with concurrent carboplatin and vincristine.',
    'Patient E, 9yo male. Relapsed bilateral Wilms tumor. Salvage ICE: carboplatin, '
    'etoposide, ifosfamide. Stem cell transplant planned if second CR achieved.',
    'Patient F, 3yo female. Neuroblastoma stage IV, MYCN amplified. High-risk induction: '
    'cisplatin, etoposide, doxorubicin, cyclophosphamide, vincristine (GPOH NB2004).',
    'Patient G, 14yo male. Osteosarcoma distal femur. MAP: methotrexate 12 g/m2, '
    'adriamycin 75 mg/m2, cisplatin 120 mg/m2. Pre-operative cycle 2.',
    'Patient H, 6yo female. AML M4. Induction: cytarabine 100 mg/m2 and daunorubicin '
    '50 mg/m2 (7+3). Lumbar puncture negative for CNS disease.',
    'Patient I, 10yo male. Ewing sarcoma pelvis. VDC/IE: vincristine, doxorubicin, '
    'cyclophosphamide alternating with ifosfamide and etoposide.',
    'Patient J, 8yo female. Wilms tumor stage I favorable histology. Right nephrectomy. '
    'Adjuvant vincristine and actinomycin-D per COG AREN0532.',
    'Patient K, 11yo male. T-cell ALL relapse. Nelarabine-based salvage. Intrathecal '
    'methotrexate CNS prophylaxis. MRD monitoring every 2 weeks.',
    'Patient L, 2yo female. Hepatoblastoma PRETEXT III. PLADO: cisplatin 80 mg/m2 and '
    'doxorubicin 60 mg/m2. Surgical resection after 4 cycles.',
    'Patient M, 15yo female. Embryonal rhabdomyosarcoma. VAC/VI: vincristine, '
    'actinomycin-D, cyclophosphamide alternating with vincristine and irinotecan.',
    'Patient N, 7yo male. Burkitt lymphoma stage IV, bone marrow involved. CODOX-M/IVAC. '
    'Tumor lysis syndrome prophylaxis: allopurinol and aggressive hydration.',
    'Patient O, 5yo male. Incidental nephrogenic rest on imaging. No active Wilms tumor. '
    'Surveillance ultrasound every 3 months per COG. No chemotherapy indicated.',
]

print(f'Corpus: {len(corpus)} clinical notes\n')
for i, note in enumerate(corpus):
    print(f'[{i:2d}] {note[:90]}...')


---
## Section 1 — Exact Text Match

The simplest possible search: is the query string present verbatim in the document?

**Strengths:** instant, zero dependencies, deterministic
**Failures:** misses `'Wilms tumour'` (British spelling), `'nephroblastoma'` (synonym), `'vincristne'` (typo)


In [ ]:
def exact_search(query, docs, case_sensitive=False):
    q = query if case_sensitive else query.lower()
    return [i for i, doc in enumerate(docs) if q in (doc if case_sensitive else doc.lower())]

for q in ['Wilms tumor', 'vincristine', 'nephroblastoma', 'kidney cancer', 'Wilms tumour']:
    hits = exact_search(q, corpus)
    print(f"'{q}': {len(hits)} hits  → {hits}")


---
## Section 2 — Regex (Regular Expressions)

Regex matches **patterns** rather than fixed strings. Essential for clinical text: drug dosages, staging, ICD codes.

**Strengths:** handles format variation, very fast
**Failures:** requires hand-crafted patterns; can't handle synonyms or typos


In [ ]:
import re

def regex_search(pattern, docs, flags=re.IGNORECASE):
    results = []
    for i, doc in enumerate(docs):
        m = re.search(pattern, doc, flags)
        if m:
            results.append((i, m.group()))
    return results

# Drug + dosage
print('--- vincristine dosage ---')
for hit_i, match in regex_search(r'vincristine\s+[\d.]+\s*mg', corpus):
    print(f'  [{hit_i}] matched: {match!r}')

# Any dosage unit
print('\n--- Any dosage ---')
for hit_i, match in regex_search(r'\d+\.?\d*\s*(?:mg/m2|g/m2|mg|Gy)', corpus)[:8]:
    print(f'  [{hit_i}] {match!r}')

# Staging
print('\n--- Staging patterns ---')
for hit_i, match in regex_search(r'stage\s+[IVX]+[AB]?', corpus):
    print(f'  [{hit_i}] {match!r}')


---
## Section 3 — Fuzzy Text Match

Fuzzy matching tolerates **typos and spelling variants** via edit-distance scoring.
`rapidfuzz.fuzz.partial_ratio` scores based on the best-matching substring.

**Strengths:** handles OCR errors, British vs. American spelling
**Failures:** purely character-level — `'nephroblastoma'` ≠ `'kidney cancer'` (different words, not a typo)


In [ ]:
from rapidfuzz import fuzz

def fuzzy_search(query, docs, threshold=75):
    results = []
    for i, doc in enumerate(docs):
        score = fuzz.partial_ratio(query.lower(), doc.lower())
        if score >= threshold:
            results.append((i, score))
    return sorted(results, key=lambda x: x[1], reverse=True)

for q in ['Wilms tumour', 'vincristne', 'nephroblstoma', 'leukaemia', 'kidney cancer']:
    hits = fuzzy_search(q, corpus)
    top = [(i, s) for i, s in hits[:2]]
    print(f"'{q}': {len(hits)} hits, top: {top}")


---
## Section 4 — TF-IDF (Term Frequency–Inverse Document Frequency)

TF-IDF maps documents to **sparse vectors**. Each dimension is a vocabulary term, weighted by:
- **TF**: how often the term appears in *this* document
- **IDF**: penalises common terms that appear in *many* documents (like "patient")

Retrieval = cosine similarity between the query vector and all document vectors.

**Strengths:** fast, no neural model needed, keyword-aware
**Failures:** `'kidney'` ≠ `'renal'`, no morphology awareness unless you add stemming


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(corpus)
print(f'TF-IDF matrix: {tfidf_matrix.shape}  (docs × terms)')

def tfidf_search(query, k=5):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_k = np.argsort(scores)[::-1][:k]
    return [(int(i), round(float(scores[i]), 4)) for i in top_k if scores[i] > 0]

for q in ['vincristine actinomycin Wilms', 'leukemia blast cells', 'kidney cancer child']:
    print(f"\nTF-IDF: '{q}'")
    for i, s in tfidf_search(q):
        print(f'  [{i}] score={s} | {corpus[i][:80]}...')


---
## Section 5 — BM25 (Best Match 25)

BM25 is the probabilistic retrieval model powering **Elasticsearch, Solr, and Lucene**.
It improves on TF-IDF with two mechanisms:

1. **TF saturation** — the 100th occurrence of a term adds far less than the 1st
2. **Document-length normalisation** — a short note that matches scores higher than a long one

**Strengths:** better ranking calibration than TF-IDF; industry default for lexical search
**Still fails:** synonyms and semantics — `'kidney cancer'` still won't find `'nephroblastoma'`


In [ ]:
from rank_bm25 import BM25Okapi

tokenised = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenised)

def bm25_search(query, k=5):
    scores = bm25.get_scores(query.lower().split())
    top_k = np.argsort(scores)[::-1][:k]
    return [(int(i), round(float(scores[i]), 4)) for i in top_k if scores[i] > 0]

# Head-to-head: TF-IDF vs BM25 on three queries
print(f"{'Query':<42} {'TF-IDF top-3':<22} BM25 top-3")
print('-' * 80)
for q in ['vincristine actinomycin Wilms', 'leukemia blast cells', 'kidney cancer child']:
    t = [i for i, _ in tfidf_search(q, k=3)]
    b = [i for i, _ in bm25_search(q, k=3)]
    print(f'{q:<42} {str(t):<22} {b}')

print('\nWilms tumor (nephroblastoma = kidney cancer) notes: [0, 4, 9, 14]')
print('Does either lexical method find them for the synonym query?')


---
## Section 6 — Semantic Search (Dense Retrieval)

A neural encoder maps text to **dense vectors** where similar *meanings* land close together.
`'kidney cancer'` and `'nephroblastoma'` appear in similar contexts during pre-training, so
their embeddings end up nearby — unlike TF-IDF/BM25 which only see character overlap.

We use **`all-MiniLM-L6-v2`** — a 80 MB sentence-transformers model, free, no API key needed.

**Strengths:** synonyms, paraphrasing, conceptual similarity
**Failures:** can miss exact alphanumeric codes/IDs that BM25 nails; slower at scale


In [ ]:
from sentence_transformers import SentenceTransformer

print('Loading model (first run downloads ~80 MB)...')
st_model = SentenceTransformer('all-MiniLM-L6-v2')
corpus_embeddings = st_model.encode(corpus, convert_to_numpy=True, show_progress_bar=True)
print(f'Embeddings shape: {corpus_embeddings.shape}  (docs × dims)')


In [ ]:
def semantic_search(query, k=5):
    q_emb = st_model.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(q_emb, corpus_embeddings).flatten()
    top_k = np.argsort(scores)[::-1][:k]
    return [(int(i), round(float(scores[i]), 4)) for i in top_k]

# The key demo — synonym query that stumps lexical methods
query = 'kidney cancer child'
print(f"Query: '{query}'  (no shared words with 'nephroblastoma')\n")
print(f"Exact:    {exact_search(query, corpus)}")
print(f"BM25:     {[i for i, _ in bm25_search(query, k=3)]}")
print(f"Semantic: {[i for i, _ in semantic_search(query, k=3)]}")
print('\nWilms tumor (nephroblastoma) indices: [0, 4, 9, 14]')
print('\n--- Semantic top-5 ---')
for i, s in semantic_search(query):
    print(f'  [{i}] score={s} | {corpus[i][:85]}...')


---
## Section 7 — Oversimplified RAG

**RAG = Retrieve → Augment → Generate**

1. **Retrieve**: find the most relevant notes (semantic search)
2. **Augment**: inject them as context in the LLM prompt
3. **Generate**: the LLM answers *grounded* in real evidence — not hallucinated

This is intentionally bare-bones. Labs 1–5 in this session add structured parsing,
evaluation, agentic re-retrieval, and knowledge graphs on top of this skeleton.


In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')


In [ ]:
from openai import OpenAI
client = OpenAI()

def simple_rag(question, k=3):
    # Step 1 — Retrieve
    hits = semantic_search(question, k=k)
    retrieved = [corpus[i] for i, _ in hits]
    context = '\n\n'.join(f'[{j+1}] {doc}' for j, doc in enumerate(retrieved))

    # Step 2 — Generate
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': (
                'You are a clinical assistant at KHCC. '
                'Answer ONLY using the provided patient notes. '
                'Cite each source as [1], [2], etc. '
                'If the answer is not in the notes, say so.'
            )},
            {'role': 'user', 'content': f'Notes:\n{context}\n\nQuestion: {question}'},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content, retrieved

questions = [
    'Which patients with kidney tumours received vincristine?',
    'What salvage regimen was used for the relapsed patient?',
    'Which patients have CNS involvement or received CNS prophylaxis?',
]

for q in questions:
    print(f'\n{"="*65}')
    print(f'Q: {q}')
    answer, sources = simple_rag(q)
    print(f'A: {answer}')
    print('Sources:')
    for j, s in enumerate(sources):
        print(f'  [{j+1}] {s[:80]}...')


---
## Summary — The Search Spectrum

Each method fixes a gap in the previous one.
In production, **hybrid search** (BM25 + semantic + a reranker) often combines their strengths.
That is exactly what modern RAG stacks like LlamaIndex (Lab 1) do under the hood.


In [ ]:
import pandas as pd

rows = [
    ('Exact Match',  'No',  'No',  'No',  'No',  'Instant',    'Known IDs / accession codes'),
    ('Regex',        'No',  'No',  'No',  'No',  'Very fast',  'Dosage / date / code patterns'),
    ('Fuzzy',        'Yes', 'No',  'No',  'No',  'Slow',       'Noisy / OCR / misspelled text'),
    ('TF-IDF',       'No',  'No',  'No',  'No',  'Fast',       'Keyword bag-of-words queries'),
    ('BM25',         'No',  'No',  'No',  'No',  'Fast',       'Full-text search engine default'),
    ('Semantic',     'Partial','Yes','Yes','No',  'Medium',     'Synonym / paraphrase queries'),
    ('RAG',          'Partial','Yes','Yes','Yes', 'Slow',       'Open-ended question answering'),
]
cols = ['Method', 'Typo-tolerant', 'Synonym-aware', 'Needs model', 'Generates answer',
        'Speed (scale)', 'Best for']
df = pd.DataFrame(rows, columns=cols)
pd.set_option('display.max_colwidth', 35)
pd.set_option('display.width', 220)
print(df.to_string(index=False))
